# Speed Calibration (Method B)

Drive at known speeds through both cameras.  
Track frame-to-frame x-displacement → derive a per-camera factor `k` (px/frame/mph).  
`estimate_speed(track_frames, camera) = median_|dx| / k`

Calibration clips live at:
```
traffic_data/calibration/[mph]/[etw|wte]/[EF|WF]/<clip>.mp4
```

Results are written to `traffic_data/calibration/speed_cal.json` and loaded by `traffic_events.ipynb`.

## Imports & Constants

In [ ]:
import os
import json
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict
import traffic_utils as tu

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"]  = "1"

PROJECT_ROOT = Path("/Users/jrill/Documents/traffic_project")
YOLO_MODEL   = PROJECT_ROOT / "yolov8n.pt"
CAL_ROOT     = PROJECT_ROOT / "traffic_data/calibration"
CAL_JSON     = CAL_ROOT / "speed_cal.json"

## Model

In [ ]:
model = tu.init_model(YOLO_MODEL)
print(f"YOLO loaded: {YOLO_MODEL.name}")

## Calibration Clip Discovery

Scans `CAL_ROOT/[mph]/[etw|wte]/[EF|WF]/` for `.mp4` files.  
Directory names at depth 1 must be integers (the known speed in mph).

In [ ]:
# Clips to skip — keyed by (cam, direction, mph).
# Add entries here for any clip where YOLO fails to track the calibration vehicle.
EXCLUDE = {
    ('EF', 'etw', 27),   # no clean track on calibration vehicle
}

CAL_RUNS = []  # (camera, direction, mph, clip_path)

for speed_dir in sorted(CAL_ROOT.iterdir(), key=lambda p: int(p.name) if p.name.isdigit() else -1):
    if not speed_dir.is_dir() or not speed_dir.name.isdigit():
        continue
    mph = int(speed_dir.name)
    for direction in ['etw', 'wte']:
        dir_path = speed_dir / direction
        if not dir_path.is_dir():
            continue
        for cam in ['EF', 'WF']:
            if (cam, direction, mph) in EXCLUDE:
                continue
            cam_path = dir_path / cam
            if not cam_path.is_dir():
                continue
            for clip in sorted(cam_path.glob('*.mp4')):
                CAL_RUNS.append((cam, direction, mph, clip))

print(f'Discovered {len(CAL_RUNS)} calibration clips ({len(EXCLUDE)} excluded):')
for cam, direction, mph, path in CAL_RUNS:
    print(f'  {cam:2s}  {direction}  @ {mph:2d} mph  ->  {path.name}')

## Pixel Velocity Computation

In [ ]:
def _track_median_dx(frames):
    dxs = []
    for i in range(1, len(frames)):
        dt = frames[i][0] - frames[i - 1][0]
        if dt:
            dxs.append(abs(frames[i][1] - frames[i - 1][1]) / dt)
    return float(np.median(dxs)) if dxs else 0.0


def compute_pixel_velocity(clip_path, min_frames=5):
    """
    Track the calibration vehicle in a clip using the same pipeline as traffic_events:
    detect_motion -> find_vehicle_windows -> run_yolo_windows -> stitch.

    Selects the track with the highest median |dx| (the moving calibration vehicle)
    and returns its median frame-to-frame x-displacement in pixels/frame.
    """
    mot_scores, _, mot_fps = tu.detect_motion(clip_path)

    if len(mot_scores):
        windows, *_ = tu.find_vehicle_windows(mot_scores, mot_fps)
    else:
        windows = []

    if not windows:
        _cap   = cv2.VideoCapture(str(clip_path))
        _total = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        _cap.release()
        windows = [(0, max(0, _total - 1))]

    track_data = tu.run_yolo_windows(clip_path, windows)

    candidates = {tid: frames for tid, frames in track_data.items() if len(frames) >= min_frames}
    if not candidates:
        raise ValueError(f'No tracks with >= {min_frames} frames in {Path(clip_path).name}')

    primary = max(candidates.values(), key=_track_median_dx)
    pv      = _track_median_dx(primary)

    if pv == 0.0:
        raise ValueError(f'No displacement data in {Path(clip_path).name}')

    return pv


def _fit_k(records):
    """Mean of (px_per_frame / mph) across a set of calibration records."""
    return float(np.mean([pv / mph for _, _, mph, pv in records]))

## Run Calibration

In [ ]:
print('Running YOLO on calibration clips...')
cal_records = []  # (camera, direction, mph, px_per_frame)
errors = []

for cam, direction, mph, clip_path in CAL_RUNS:
    try:
        pv = compute_pixel_velocity(clip_path)
        cal_records.append((cam, direction, mph, pv))
        print(f'  {cam:2s}  {direction}  @ {mph:2d} mph  ->  {pv:.2f} px/frame')
    except Exception as e:
        errors.append((cam, direction, mph, clip_path.name, str(e)))
        print(f'  {cam:2s}  {direction}  @ {mph:2d} mph  ->  ERROR: {e}')

if errors:
    print(f'\n{len(errors)} clip(s) failed.')

In [ ]:
import matplotlib.pyplot as plt

COMBOS = [('EF', 'etw'), ('WF', 'etw'), ('EF', 'wte'), ('WF', 'wte')]
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

fig, ax = plt.subplots(figsize=(9, 5))

for (cam, direction), color in zip(COMBOS, COLORS):
    points = sorted(
        [(mph, pv) for c, d, mph, pv in cal_records if c == cam and d == direction],
        key=lambda t: t[0],
    )
    if points:
        xs, ys = zip(*points)
        ax.plot(xs, ys, marker='o', color=color, label=f'{cam} {direction}')

ax.set_xlabel('Speed (mph)')
ax.set_ylabel('Pixel velocity (px/frame)')
ax.set_title('Speed Calibration — pixel velocity vs. known speed')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Fit Calibration Factors & Save

In [ ]:
# Build SPEED_CAL[camera][direction] = k (px/frame/mph)
# 'avg' key = camera-wide average used as fallback when direction has no data.
SPEED_CAL = {}
by_cam = defaultdict(list)
for rec in cal_records:
    by_cam[rec[0]].append(rec)

for cam, recs in by_cam.items():
    by_dir = defaultdict(list)
    for rec in recs:
        by_dir[rec[1]].append(rec)
    SPEED_CAL[cam] = {d: _fit_k(dr) for d, dr in by_dir.items()}
    SPEED_CAL[cam]['avg'] = _fit_k(recs)

print('Calibration factors (px/frame/mph):')
for cam, factors in sorted(SPEED_CAL.items()):
    for key, k in sorted(factors.items()):
        print(f'  {cam} [{key}]: {k:.4f}')

print('\nValidation (round-trip):')
for cam, direction, mph, pv in cal_records:
    k   = SPEED_CAL[cam].get(direction) or SPEED_CAL[cam]['avg']
    est = pv / k
    err = est - mph
    print(f'  {cam:2s}  {direction}  @ {mph:2d} mph  ->  estimated {est:.1f} mph  ({err:+.1f})')

with open(CAL_JSON, 'w') as f:
    json.dump(SPEED_CAL, f, indent=2)
print(f'\nSaved -> {CAL_JSON}')